# Normal map bug

The normal map result depends on the order of intersectors. Each intersector has its own z-buffer, which is not shared, while all write to the same GPU normal map buffer. As a result, later kernels overwrite values from earlier ones, causing incorrect normals.

## Resulting images

### Normal map: Order spheres, cylinders, cuboids

<img src="./output/test_exited_with_error/normal.png" width="500"/>

### Normal map: Order cuboids, spheres, cylinders

<img src="./output/test_exited_with_error/second_normal.png" width="500"/>


## Analysis

- The normal map appears to come from the first intersector, but the actual cause is different.

- All intersectors write to the same GPU memory buffer for the normal map.

- Each intersector has its own z-buffer, which is not shared and is reset at the start of each kernel.

- Kernels run sequentially, so the last kernel overwrites previous values.

**Conclusion**: The bug occurs because z-buffers are isolated per intersector, leading to overwrites in the shared normal map buffer.

## Bug Fix and Code Changes
### Goal

Ensure the z-buffer is shared across all intersector kernels so that the correct values are reliably written into the normal map buffer.

### Changes
#### Intersectors

- Removed individual z-buffers from each intersector.

- Updated the signatures of the intersect methods to accept a shared z-buffer parameter.

#### GPUMappedObject

- Added a method to initialize the z-buffer on both host and device memory.

#### HeightFieldExtractor

- Introduced the z-buffer as a member of HeightFieldExtractor.

- Initialized the z-buffer in the constructor.

- Reset the z-buffer to empty on each call to extract_data_representation().

- Passed the shared z-buffer to all kernel calls.

## Resulting images after code changes

### Normal map: Order spheres, cylinders, cuboids

<img src="./output/normal.png" width="500"/>

### Normal map: Order cuboids, spheres, cylinders

<img src="./output/second_normal.png" width="500"/>


In [ ]:
import pytest
import timeit
import os
import numpy as np
from PIL import Image
import numpy.lib.recfunctions as rf
from scipy.spatial.transform import Rotation
os.add_dll_directory(r"C:\Program Files\NVIDIA GPU Computing Toolkit\CUDA\v12.9\bin")
from _preprocess_module import HeightFieldExtractor
import gc

## Test the normal map generation and save the results

In [ ]:
cylinders = np.empty( (1, 9), dtype=np.float32 )
cuboids = np.empty( (1, 10), dtype=np.float32 )
spheres = np.empty((1,4),  dtype=np.float32 )
rotation = 3.14159265359 / 2.0


rotationY = Rotation.from_rotvec([45.0, 0.0,   0.0], degrees=True)
rotationX = Rotation.from_rotvec([ 0.0, 45.0,  0.0], degrees=True)
rotationZ = Rotation.from_rotvec([ 0.0,  0.0, 45.0], degrees=True)
qx = rotationX.as_quat()
qy = rotationY.as_quat()
qz = rotationZ.as_quat()

cylinders[0] = [ 425.0, 225.0,  500.0, qx[0], qx[1], qx[2], qx[3], 100.0, 200.0 ]
cuboids[0] = [ 180.0, 180.0, 80.0, qx[0], qx[1], qx[2], qx[3], 20.0, 40.0, 80.0 ]
spheres[0] = [ 200.0, 160.0, 40.0, 30.0]



print("Generate first normal map")

preprocessor = HeightFieldExtractor( (850,850), 2, 256 )
preprocessor.add_spheres( spheres )
preprocessor.add_cylinders( cylinders )
preprocessor.add_cuboids( cuboids )

extended_heightfield, normal_map = preprocessor.extract_data_representation( 0.0 )

# Save the extended heightfields
for z in range(extended_heightfield.shape[2]):
    # entry_0 = rf.structured_to_unstructured(extended_heightfield[:,:,z]);
    entry = extended_heightfield[:,:,z]
    entry = entry.astype(np.uint16)
    img = Image.fromarray(entry, "I;16")
    img.save("output/integrated_"+str(z)+".tif")

# Convert the first normal map
normal_map = normal_map.squeeze(2)
normal_map = rf.structured_to_unstructured( normal_map )
normal_map = ( normal_map + 1.0 ) * 127.5
normal = normal_map.astype(np.uint8)



print("Generate second normal map")

preprocessor_2 = HeightFieldExtractor( (850,850), 2, 256 )
preprocessor_2.add_cuboids( cuboids )
preprocessor_2.add_spheres( spheres )
preprocessor_2.add_cylinders( cylinders )

extended_heightfield_2, normal_map_2 = preprocessor_2.extract_data_representation( 0.0 )

# Save the extended heightfields
for z in range(extended_heightfield_2.shape[2]):
    # entry_0 = rf.structured_to_unstructured(extended_heightfield[:,:,z]);
    entry = extended_heightfield_2[:,:,z]
    entry = entry.astype(np.uint16)
    img = Image.fromarray(entry, "I;16")
    img.save("output/second_integrated_"+str(z)+".tif")

# Convert the second normal map
normal_map_2 = normal_map_2.squeeze(2)
normal_map_2 = rf.structured_to_unstructured( normal_map_2 )
normal_map_2 = ( normal_map_2 + 1.0 ) * 127.5
normal_2 = normal_map_2.astype(np.uint8)

# Save both generated images

img = Image.fromarray(normal, "RGB")
img.save("output/normal.tif")

img_2 = Image.fromarray(normal_2, "RGB")
img_2.save("output/second_normal.tif")

assert np.array_equal(normal, normal_2), "Arrays are not equal"
del preprocessor
del preprocessor_2



## Test the normal map generation and validate it using pytest. Afterwards run a benchmark using timeit.

In [ ]:
%%writefile test_heightfield.py


import pytest
import timeit
import os
import numpy as np
from PIL import Image
import numpy.lib.recfunctions as rf
from scipy.spatial.transform import Rotation

os.add_dll_directory(r"C:\Program Files\NVIDIA GPU Computing Toolkit\CUDA\v12.9\bin")
from _preprocess_module import HeightFieldExtractor


def create_test_geometry():
    cylinders = np.empty((1, 9), dtype=np.float32)
    cuboids = np.empty((1, 10), dtype=np.float32)
    spheres = np.empty((1, 4), dtype=np.float32)

    rotationX = Rotation.from_rotvec([0.0, 45.0, 0.0], degrees=True)
    qx = rotationX.as_quat()

    cylinders[0] = [425.0, 225.0, 500.0, qx[0], qx[1], qx[2], qx[3], 100.0, 200.0]

    cuboids[0] = [180.0, 180.0, 80.0, qx[0], qx[1], qx[2], qx[3], 20.0, 40.0, 80.0]

    spheres[0] = [200.0, 160.0, 40.0, 30.0]

    return spheres, cylinders, cuboids


def run_heightfield_extraction():
    spheres, cylinders, cuboids = create_test_geometry()

    preprocessor = HeightFieldExtractor((850, 850), 2, 256)
    preprocessor.add_spheres(spheres)
    preprocessor.add_cylinders(cylinders)
    preprocessor.add_cuboids(cuboids)

    _, normal_map = preprocessor.extract_data_representation(0.0)

    normal_map = normal_map.squeeze(2)
    normal_map = rf.structured_to_unstructured(normal_map)
    normal_map = (normal_map + 1.0) * 127.5
    normal = normal_map.astype(np.uint8)

    del preprocessor
    return normal


def test_heightfield_normal_map_order_independence():
    spheres, cylinders, cuboids = create_test_geometry()

    # --- First order ---
    preprocessor = HeightFieldExtractor((850, 850), 2, 256)
    preprocessor.add_spheres(spheres)
    preprocessor.add_cylinders(cylinders)
    preprocessor.add_cuboids(cuboids)

    _, normal_map = preprocessor.extract_data_representation(0.0)

    normal_map = normal_map.squeeze(2)
    normal_map = rf.structured_to_unstructured(normal_map)
    normal_map = (normal_map + 1.0) * 127.5
    normal = normal_map.astype(np.uint8)

    # --- Second order ---
    preprocessor_2 = HeightFieldExtractor((850, 850), 2, 256)
    preprocessor_2.add_cuboids(cuboids)
    preprocessor_2.add_spheres(spheres)
    preprocessor_2.add_cylinders(cylinders)

    _, normal_map_2 = preprocessor_2.extract_data_representation(0.0)

    normal_map_2 = normal_map_2.squeeze(2)
    normal_map_2 = rf.structured_to_unstructured(normal_map_2)
    normal_map_2 = (normal_map_2 + 1.0) * 127.5
    normal_2 = normal_map_2.astype(np.uint8)

    del preprocessor
    del preprocessor_2
    assert np.array_equal(normal, normal_2), (
        "Normal maps differ depending on insertion order"
    )


In [ ]:
!pytest
import test_heightfield
# CUDA warm up
for _ in range(3):
    test_heightfield.run_heightfield_extraction()
gc.disable
%timeit -n 30 -r 3 test_heightfield.run_heightfield_extraction()
